# UAV GPS spoofing & jamming detection

**Kościuszkon 2026 — Honeywell Theme #2**

GPS is the only absolute-position sensor on most drones. A SDR like HackRF can
emit a stronger fake constellation, smoothly walking the receiver away from its
true position (spoofing), or just drowning the L1 band in noise (jamming).

This notebook trains a RandomForest on GPS receiver telemetry and compares it
against a physics-rules baseline, on the *Live GPS Spoofing and Jamming*
dataset (IEEE DataPort, ~10k samples). When the raw CSVs are not available
locally we fall back to a labelled simulation that exhibits the same physical
signatures (drift, HDOP/VDOP spikes, satellite drop-out) so the pipeline stays
reproducible.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

sys.path.append(str(pathlib.Path.cwd().parent))
from gps_detection_utils import train_rf, evaluate, plot_results, compare_methods

RNG = np.random.default_rng(42)
plt.rcParams['figure.dpi'] = 90

## 1. Data — real CSV or labelled simulation

We try the real IEEE dataset first. If it isn't on disk we generate a
**time-ordered** flight with discrete attack windows: a smooth circular path
plus realistic 1 m GPS noise during normal flight, position drift during
spoofing windows, and HDOP/VDOP spikes plus satellite drop-out during jamming
windows. Generating attacks as *segments* (not random rows) keeps the temporal
structure intact, so position-derivative features remain physically meaningful.

In [ ]:
def simulate_uav_flight(n=10067, attack_rate=0.194, rng=RNG):
    """Simulate a drone flight with discrete spoofing/jamming windows.

    Returns a time-ordered DataFrame whose feature distributions overlap
    between classes (so the task isn't trivial)."""
    t = np.arange(n)
    base_lat = 40.7128 + 5e-4 * np.sin(2 * np.pi * t / 2000)
    base_lon = -74.0060 + 1e-3 * np.sin(2 * np.pi * t / 1000)
    base_alt = 100 + 8 * np.sin(2 * np.pi * t / 500)

    is_attack = np.zeros(n, dtype=bool)
    atk_type  = np.array(['normal'] * n, dtype=object)
    target = int(n * attack_rate)
    while is_attack.sum() < target:
        start = int(rng.integers(0, n - 400))
        length = int(rng.integers(120, 350))
        kind = rng.choice(['spoofing', 'jamming'], p=[0.6, 0.4])
        is_attack[start:start + length] = True
        atk_type[start:start + length] = kind

    lat = base_lat + rng.normal(0, 1e-5, n)
    lon = base_lon + rng.normal(0, 1e-5, n)
    alt = base_alt + rng.normal(0, 0.5, n)
    vel_n = np.gradient(base_lat) * 111_320 + rng.normal(0, 0.3, n)
    vel_e = np.gradient(base_lon) * 111_320 * np.cos(np.radians(base_lat)) + rng.normal(0, 0.3, n)
    vel_d = np.gradient(base_alt) + rng.normal(0, 0.1, n)

    eph  = np.abs(rng.normal(2.0, 0.5, n))
    epv  = np.abs(rng.normal(3.0, 0.7, n))
    hdop = np.abs(rng.normal(1.5, 0.3, n))
    vdop = np.abs(rng.normal(2.0, 0.4, n))
    sats = rng.integers(8, 14, n).astype(float)
    jam_ind = np.zeros(n)

    spoof = atk_type == 'spoofing'
    if spoof.any():
        # accumulating fake-position drift: the give-away of a smooth spoof
        lat[spoof] += np.cumsum(rng.normal(0, 4e-6, spoof.sum()))
        lon[spoof] += np.cumsum(rng.normal(0, 4e-6, spoof.sum()))
        vel_n[spoof] += rng.normal(0, 1.5, spoof.sum())
        vel_e[spoof] += rng.normal(0, 1.5, spoof.sum())

    jam = atk_type == 'jamming'
    if jam.any():
        k = jam.sum()
        hdop[jam] *= rng.uniform(2, 4, k)
        vdop[jam] *= rng.uniform(2, 4, k)
        eph[jam]  *= rng.uniform(1.5, 3, k)
        epv[jam]  *= rng.uniform(1.5, 3, k)
        sats[jam] = np.clip(sats[jam] - rng.integers(2, 5, k), 3, 14)
        jam_ind[jam] = rng.choice([0, 1], k, p=[0.3, 0.7])

    return pd.DataFrame({
        'timestamp': pd.date_range('2024-01-01', periods=n, freq='1s'),
        'recording': 'sim_flight',  # one continuous session
        'lat_y': lat, 'lon_y': lon, 'alt_y': alt,
        'vel_n_m_s': vel_n, 'vel_e_m_s': vel_e, 'vel_d_m_s': vel_d,
        'eph_y': eph, 'epv_y': epv, 'hdop': hdop, 'vdop': vdop,
        'satellites_used': sats, 'jamming_indicator': jam_ind,
        'attack_type': atk_type,
        'is_attack': is_attack.astype(int),
    })


def load_uav():
    """Load real IEEE dataset if present, else simulate."""
    spoof_csv = '../../UAVAttackData/Live GPS Spoofing and Jamming/GPS Spoofing/Processed/spoofing-merged-gps-only.csv'
    jam_csv   = '../../UAVAttackData/Live GPS Spoofing and Jamming/GPS Jamming/Processed/jamming-merged-gps-only.csv'
    try:
        sp = pd.read_csv(spoof_csv);  sp['attack_type'] = 'spoofing'; sp['recording'] = 'spoofing_csv'
        jm = pd.read_csv(jam_csv);    jm['attack_type'] = 'jamming';  jm['recording'] = 'jamming_csv'
        df = pd.concat([sp, jm], ignore_index=True)
        df['is_attack'] = (df['label'] == 'malicious').astype(int)
        print(f"Loaded real IEEE dataset: {len(df):,} samples")
    except FileNotFoundError:
        df = simulate_uav_flight()
        print(f"Real CSVs not found — using labelled simulation: {len(df):,} samples")
    return df

df = load_uav()
df.head(3)

## 2. Exploratory data analysis

In [ ]:
print(f"Shape:           {df.shape}")
print(f"Missing values:  {int(df.isna().sum().sum())}")
print(f"Attack rate:     {df['is_attack'].mean():.1%}")
print(f"Attack types:    {df['attack_type'].value_counts().to_dict()}")
print("\nSummary statistics for selected GPS quality features:")
df[['hdop', 'vdop', 'satellites_used', 'eph_y', 'epv_y']].describe().round(2)

**What separates attack from normal?** Below: class balance, HDOP shift under
jamming, satellite drop-out, and per-step position movement (spoofing signature).

In [ ]:
# For the EDA-only plot we need per-step distance — compute it within each
# recording so we don't get spurious jumps at file boundaries.
g = df.groupby('recording', sort=False)
step_m = np.hypot(g['lat_y'].diff(), g['lon_y'].diff()).fillna(0) * 111_320

normal = df['is_attack'] == 0
fig, ax = plt.subplots(2, 2, figsize=(11, 7))

ax[0, 0].pie(df['is_attack'].value_counts().sort_index(),
             labels=['benign', 'attack'], autopct='%1.1f%%',
             colors=['#7fbf7f', '#d65f5f'])
ax[0, 0].set_title('Class balance')

ax[0, 1].hist([df.loc[normal, 'hdop'], df.loc[~normal, 'hdop']],
              bins=40, label=['benign', 'attack'], color=['#7fbf7f', '#d65f5f'])
ax[0, 1].set_title('HDOP distribution');  ax[0, 1].set_xlabel('HDOP'); ax[0, 1].legend()

ax[1, 0].hist([df.loc[normal, 'satellites_used'], df.loc[~normal, 'satellites_used']],
              bins=range(3, 16), label=['benign', 'attack'], color=['#7fbf7f', '#d65f5f'])
ax[1, 0].set_title('Satellites used'); ax[1, 0].set_xlabel('count'); ax[1, 0].legend()

ax[1, 1].hist([step_m[normal].clip(0, 5), step_m[~normal].clip(0, 5)],
              bins=40, label=['benign', 'attack'], color=['#7fbf7f', '#d65f5f'])
ax[1, 1].set_title('Per-step position move (m, clipped at 5)')
ax[1, 1].set_xlabel('metres / sample'); ax[1, 1].legend()

plt.tight_layout(); plt.show()

## 3. Feature engineering

Two principles:

* **Compute derivatives in time order** — sort by timestamp first, then take
  consecutive differences. Computing `diff()` on already-shuffled rows gives
  noise; computing it grouped by `attack_type` would *leak the label* into
  every derivative feature.
* **Reach for physical quantities** — `speed_3d`, per-step position move,
  GPS-quality aggregate. These are the signals a domain expert would inspect
  by hand.

In [ ]:
# Take consecutive differences within each `recording` (the simulator has one
# session, the real dataset concatenates two CSV files). Grouping by
# `recording` is *not* label leakage — the per-row label is `is_attack`,
# while `recording` is a session identifier.
g = df.groupby('recording', sort=False)
df['speed_3d']    = np.linalg.norm(df[['vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s']], axis=1)
df['lat_delta']   = g['lat_y'].diff().abs().fillna(0)
df['lon_delta']   = g['lon_y'].diff().abs().fillna(0)
df['alt_delta']   = g['alt_y'].diff().abs().fillna(0)
df['step_metres'] = np.hypot(df['lat_delta'], df['lon_delta']) * 111_320
df['gps_quality'] = df['hdop'] + df['vdop']

FEATURES = ['speed_3d', 'step_metres', 'alt_delta',
            'eph_y', 'epv_y', 'hdop', 'vdop', 'gps_quality',
            'satellites_used', 'jamming_indicator']
print(f"{len(FEATURES)} features used")
df[FEATURES].describe().round(2)

## 4. Methods

* **Baseline (physics rules)**: an alarm fires if any one threshold is
  crossed — speed > 55 m/s, single-step move > 5 m, or GPS-quality > 6.
  Honest, transparent, easy to audit, but conservative.
* **RandomForest (100 trees, depth 12, balanced classes)**: learns
  joint distributions over the same telemetry. Stratified 70/30 split.

In [ ]:
# Physics-rules baseline (vectorised)
df['rule_alarm'] = (
    (df['speed_3d']    > 55) |
    (df['step_metres'] > 5)  |
    (df['gps_quality'] > 6)
).astype(int)
rule_acc = (df['rule_alarm'] == df['is_attack']).mean()
print(f"Rules-only accuracy:  {rule_acc:.4f}")

# RandomForest on the same telemetry
model, X_te, y_te = train_rf(df, FEATURES, 'is_attack')
ml_metrics, y_pred, _ = evaluate(model, X_te, y_te, label="UAV — RandomForest on test set")

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plot_results(y_te, y_pred, importance, ['benign', 'attack'], title='UAV')
print("\nClassification report on test set:")
print(classification_report(y_te, y_pred, target_names=['benign', 'attack']))

## 5. Comparison & honest interpretation

In [ ]:
compare_methods(ml_metrics, rule_acc, len(df), df['is_attack'].mean(), domain="UAV")

**Interpretation.** The rule baseline catches the obvious cases (large position
jumps, severe HDOP/VDOP spikes) but misses subtle drift attacks where each
step looks plausible. RandomForest combines the same signals jointly and
recovers most of those — the per-step move and GPS-quality aggregate
dominate Gini importance, matching the physical intuition.

**Trade-offs.** Rules are auditable and explainable line-by-line; ML gives
better recall but is harder to certify. The two are complementary: rules as
a fail-safe, ML as a finer detector.

**Limitations.** Numbers above are on the labelled simulation; real-world
attacks include cases the simulator does not cover (e.g. lift-off / takeover
spoofs that smoothly match initial conditions, or partial jamming that
spares only a few satellites). The shape of the comparison is informative,
the absolute headline number is not.